In [7]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import cv2
import mediapipe as mp
import csv
import os
from datetime import datetime
from itertools import combinations
from scipy.signal import savgol_filter

ВСЕ НУЖНЫЕ НАМ ФУНКЦИИ

In [8]:
# Обработка видео с сохранением

def process_video_facemesh(video_path, csv_dir, video_output_dir):
    
    # Инициализация MediaPipe
    mp_drawing = mp.solutions.drawing_utils
    mp_drawing_styles = mp.solutions.drawing_styles
    mp_face_mesh = mp.solutions.face_mesh
    
    print(f"Обрабатываю файл: {video_path}")
    
    # Создаем директории, если они не существуют
    os.makedirs(csv_dir, exist_ok=True)
    os.makedirs(video_output_dir, exist_ok=True)
    
    # Формируем пути для выходных файлов
    video_filename = os.path.basename(video_path)
    csv_filename = os.path.splitext(video_filename)[0] + '_face_landmarks.csv'
    output_video_filename = os.path.splitext(video_filename)[0] + '_processed.mp4'
    
    csv_path = os.path.join(csv_dir, csv_filename)
    output_video_path = os.path.join(video_output_dir, output_video_filename)
    
    print(f"CSV файл будет сохранен: {csv_path}")
    print(f"Обработанное видео будет сохранено: {output_video_path}")
    
    drawing_spec = mp_drawing.DrawingSpec(thickness=1, circle_radius=1)
    cap = cv2.VideoCapture(video_path)
    
    # Получаем параметры видео для сохранения
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_count = 0
    
    # Инициализируем VideoWriter для сохранения обработанного видео
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))
    
    # Создаем CSV файл и записываем заголовки
    with open(csv_path, 'w', newline='', encoding='utf-8') as csvfile:
        csv_writer = csv.writer(csvfile)
        csv_writer.writerow(['frame_number', 'timestamp_seconds', 'landmark_index', 'x', 'y', 'z'])
    
    with mp_face_mesh.FaceMesh(
        max_num_faces=1,
        refine_landmarks=True,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5) as face_mesh:
        
        while cap.isOpened():
            success, image = cap.read()
            if not success:
                print("Конец видео или ошибка загрузки кадра.")
                break
            
            # Рассчитываем временную метку
            timestamp = frame_count / fps if fps > 0 else frame_count
            
            # Чтобы улучшить производительность, помечаем изображение как доступное только для чтения
            image.flags.writeable = False
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            results = face_mesh.process(image)
            
            # Рисуем аннотации сетки лица на изображении
            image.flags.writeable = True
            image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
            
            # Открываем CSV файл для добавления данных каждого кадра
            with open(csv_path, 'a', newline='', encoding='utf-8') as csvfile:
                csv_writer = csv.writer(csvfile)
                
                if results.multi_face_landmarks:
                    for face_landmarks in results.multi_face_landmarks:
                        # Записываем координаты каждой точки лица
                        for landmark_idx, landmark in enumerate(face_landmarks.landmark):
                            csv_writer.writerow([
                                frame_count,
                                round(timestamp, 3),
                                landmark_idx,
                                round(landmark.x, 6),
                                round(landmark.y, 6),
                                round(landmark.z, 6)
                            ])
                        
                        # Рисуем landmarks на изображении
                        mp_drawing.draw_landmarks(
                            image=image,
                            landmark_list=face_landmarks,
                            connections=mp_face_mesh.FACEMESH_TESSELATION,
                            landmark_drawing_spec=None,
                            connection_drawing_spec=mp_drawing_styles
                            .get_default_face_mesh_tesselation_style())
                        mp_drawing.draw_landmarks(
                            image=image,
                            landmark_list=face_landmarks,
                            connections=mp_face_mesh.FACEMESH_CONTOURS,
                            landmark_drawing_spec=None,
                            connection_drawing_spec=mp_drawing_styles
                            .get_default_face_mesh_contours_style())
                        mp_drawing.draw_landmarks(
                            image=image,
                            landmark_list=face_landmarks,
                            connections=mp_face_mesh.FACEMESH_IRISES,
                            landmark_drawing_spec=None,
                            connection_drawing_spec=mp_drawing_styles
                            .get_default_face_mesh_iris_connections_style())
                else:
                    # Если лицо не обнаружено, записываем пустые значения для всех landmarks
                    for landmark_idx in range(478):  # MediaPipe Face Mesh имеет 478 landmarks
                        csv_writer.writerow([
                            frame_count,
                            round(timestamp, 3),
                            landmark_idx,
                            '', '', ''  # Пустые значения для координат
                        ])
            
            # Сохраняем обработанный кадр в видео
            out.write(image)
            
            # Показываем изображение
            cv2.imshow('MediaPipe Face Mesh', image)
            
            # Выводим прогресс обработки
            print(f"Обработан кадр {frame_count}, время: {timestamp:.2f} сек")
            
            frame_count += 1
            
            if cv2.waitKey(5) & 0xFF == 27:  # ESC для выхода
                break
    
    cap.release()
    out.release()  # Не забываем освободить VideoWriter
    cv2.destroyAllWindows()
    
    print(f"\nОбработка завершена!")
    print(f"Всего обработано кадров: {frame_count}")
    print(f"CSV файл сохранен: {csv_path}")
    print(f"Обработанное видео сохранено: {output_video_path}")

In [9]:
def transform_landmarks_csv(input_csv_path, output_csv_path=None, delete_if_empty=True):
    
    # Если выходной путь не указан, перезаписываем исходный
    if output_csv_path is None:
        output_csv_path = input_csv_path
    
    # Проверяем существование файла
    if not os.path.exists(input_csv_path):
        print(f"Файл не существует: {input_csv_path}")
        return None, None
    
    # Проверяем размер файла
    file_size = os.path.getsize(input_csv_path)
    if file_size == 0:
        print(f"Файл пустой (0 байт): {input_csv_path}")
        if delete_if_empty:
            os.remove(input_csv_path)
            print(f"Файл удален: {input_csv_path}")
        return None, None
    
    try:
        # Чтение исходного файла
        df = pd.read_csv(input_csv_path)
        
        # Проверяем, пустой ли DataFrame
        if df.empty:
            print(f"CSV файл пустой (0 строк): {input_csv_path}")
            if delete_if_empty:
                os.remove(input_csv_path)
                print(f"Файл удален: {input_csv_path}")
            return None, None
        
        print(f"Обработка файла: {input_csv_path}")
        print(f"Исходный размер: {df.shape}")
        
        # Проверяем наличие необходимых колонок
        required_columns = ['frame_number', 'timestamp_seconds', 'x', 'y', 'z']
        missing_columns = [col for col in required_columns if col not in df.columns]
        if missing_columns:
            print(f"Файл имеет неправильный формат. Отсутствуют колонки: {missing_columns}")
            print(f"Пропускаем файл: {input_csv_path}")
            return None, None
        
        # Создание мультииндекса для колонок
        columns = []
        for coord in ['x', 'y', 'z']:
            for i in range(478):
                columns.append(f"{coord}_{i}")
        
        # Группировка по фреймам и создание новых колонок
        grouped = df.groupby(['frame_number', 'timestamp_seconds']).agg({
            'x': lambda x: list(x),
            'y': lambda x: list(x),
            'z': lambda x: list(x)
        }).reset_index()
        
        # Проверяем, есть ли данные после группировки
        if grouped.empty:
            print(f"Нет данных после группировки (0 фреймов): {input_csv_path}")
            if delete_if_empty:
                os.remove(input_csv_path)
                print(f"Файл удален: {input_csv_path}")
            return None, None
        
        # Разворачивание списков в отдельные колонки
        expanded_data = []
        for _, row in grouped.iterrows():
            frame_data = [row['frame_number'], row['timestamp_seconds']]
            # Добавляем координаты в правильном порядке (0-477)
            for i in range(478):
                frame_data.append(row['x'][i])
            for i in range(478):
                frame_data.append(row['y'][i])
            for i in range(478):
                frame_data.append(row['z'][i])
            expanded_data.append(frame_data)
        
        # Создание нового DataFrame
        new_df = pd.DataFrame(
            expanded_data,
            columns=['frame_number', 'timestamp_seconds'] + columns
        )
        
        # Сохранение результата
        new_df.to_csv(output_csv_path, index=False)
        
        print("Преобразование завершено!")
        print(f"Новая таблица имеет размер: {new_df.shape}")
        print(f"Количество фреймов: {len(new_df)}")
        print(f"Количество колонок: {len(new_df.columns)}")
        print(f"Сохранено в: {output_csv_path}")
        
        return new_df, output_csv_path
        
    except pd.errors.EmptyDataError:
        # Если файл пустой или содержит только заголовки
        print(f"CSV файл пустой или содержит только заголовки: {input_csv_path}")
        if delete_if_empty:
            os.remove(input_csv_path)
            print(f"Файл удален: {input_csv_path}")
        return None, None
        
    except Exception as e:
        print(f"Ошибка при обработке файла {input_csv_path}: {e}")
        return None, None

In [10]:
def dist_beetw_eyes(input_csv_path):

    df = pd.read_csv(input_csv_path)
    output_csv_path = f"{input_csv_path}"
    col1_x = 'x_467'  
    col2_x = 'x_472'  
    col1_y = 'y_467'
    col2_y = 'y_472'
    col1_z = 'z_467'
    col2_z = 'z_472'

    distances = []

    for index, row in df.iterrows():
        #Первая точка
        x1 = row[col1_x]
        y1 = row[col1_y]
        z1 = row[col1_z]
        #Вторая точка
        x2 = row[col2_x]
        y2 = row[col2_y]
        z2 = row[col2_z]
        # Вычисляем евклидово расстояние между двумя точками
        distance = np.sqrt((x2 - x1)**2 + (y2 - y1)**2 + (z2 - z1)**2)
        distances.append(distance)

    # Добавляем расстояния как новую колонку в DataFrame
    df['distance_eyes'] = distances

    # Сохраняем результат с новой колонкой
    df.to_csv(output_csv_path, index=False)

    print(f"\nДобавлена колонка 'distance_eyes' с расстоянием между точками 468 и 473 в {input_csv_path}")
    
    return distances

In [11]:
# ФИЧИ:
# Упражнение - поднять брови, лобная мышца:
brow_1_l = [105,66,107]
brow_1_r = [334,296,336]
forehead = [10,10,10]

# Упражнение - поднять брови, депрессор бровей:
brow_2_l = [63, 105, 66, 107]
eyelid_l = [160, 159, 158, 157]
brow_2_r = [293, 334, 296, 336]
eyelid_r = [387,386,385,384]


brow_3_l = [52, 53, 46]
point_l = [107, 107, 107]
brow_3_r = [282, 283, 276]
point_r = [336, 336, 336, 336]
brow_4_l = [107,55]
brow_4_r = [336,285]
bridge = [168, 168]


In [12]:
import pandas as pd
import numpy as np

def calculate_distances_flexible(input_csv_path, pairs_list=None):
    """
    Вычисляет евклидовы расстояния между заданными парами точек
    
    Parameters:
    -----------
    input_csv_path : str
        Путь к CSV файлу с данными
    pairs_list : list of tuples, optional
        Список пар для вычисления. Если None, используется список по умолчанию
    
    Returns:
    --------
    df_full : pd.DataFrame
        Исходный DataFrame с добавленными колонками расстояний
    """
    output_csv_path = input_csv_path
    df = pd.read_csv(input_csv_path)
    print(f"Обрабатываю : {input_csv_path}")
    
    # output_csv_path = input_csv_path
    df_full = df.copy()
    
    # Список пар по умолчанию на основе ваших массивов
    if pairs_list is None:
        pairs_list = [
            # Упражнение 1: поднять брови, лобная мышца
            (105, 10), (66, 10), (107, 10),
            (336, 10), (296, 10), (334, 10),
            
            # Упражнение 2: поднять брови, депрессор бровей
            (105, 4), (63, 4), (66, 4), (107, 4),
            (336, 4), (296, 4), (334, 4), (293, 4),
            
            # Упражнение 3
            (52, 107), (53, 107), (46, 107),
            (282, 336), (283, 336), (276, 336),
            
            # Упражнение 4
            (107, 168), (55, 168),
            (336, 168), (285, 168)
        ]
    
    # Удаляем дубликаты
    pairs_list = list(set(pairs_list))
    
    print(f"\nВычисляю расстояния для {len(pairs_list)} пар точек...")
    print("Пары:", pairs_list)
    print()
    
    # Вычисляем расстояния для каждой пары
    for i, j in pairs_list:
        try:
            x_i = df[f'x_{i}'].values
            y_i = df[f'y_{i}'].values
            z_i = df[f'z_{i}'].values
            
            x_j = df[f'x_{j}'].values
            y_j = df[f'y_{j}'].values
            z_j = df[f'z_{j}'].values
            
            # Вычисляем евклидово расстояние в 3D
            raw_distance = np.sqrt((x_i - x_j)**2 + (y_i - y_j)**2 + (z_i - z_j)**2)
            
            # Нормировка на расстояние между глазами
            if 'distance_eyes' in df.columns:
                distances = raw_distance / df['distance_eyes'].values
                norm_text = " (нормировано)"
            else:
                distances = raw_distance
                norm_text = " (без нормировки)"
            
            # Добавляем в результирующий датафрейм
            column_name = f'dist_{i}_{j}'
            df_full[column_name] = distances
            print(f"✓ {column_name}: точки {i} → {j}{norm_text}")
            
        except KeyError as e:
            print(f"✗ Ошибка для пары ({i}, {j}): отсутствует колонка {e}")
        except Exception as e:
            print(f"✗ Ошибка для пары ({i}, {j}): {e}")
    
    # Сохраняем результат
    df_full.to_csv(output_csv_path, index=False)
    print(f"\n✅ Результат сохранен в: {output_csv_path}")
    
    return df_full

In [13]:
def calculate_statistics1(input_csv_path, distance_prefix='dist_'):

    df = pd.read_csv(input_csv_path)
    print(f"Обрабатываю : {input_csv_path}")

    if '_with_eyes.csv' in input_csv_path:
        input_csv_path = input_csv_path.replace('_with_eyes.csv', '')
    output_csv_path = input_csv_path.replace('face_landmarks', 'statistics')
    
    distance_cols = [col for col in df.columns if col.startswith(distance_prefix)]
    
    if not distance_cols:
        raise ValueError(f"Не найдено столбцов с префиксом '{distance_prefix}'")
    
    print(f"Найдено {len(distance_cols)} столбцов с расстояниями")
    
    statistics = []
    
    for col in distance_cols:
        values = df[col].values
        
        # статистики
        stats = {
            'pair': col,
            'min': np.min(values),
            'max': np.max(values),
            'std': np.std(values),
            'mean': np.mean(values),
            'median': np.median(values),
            'range': np.max(values) - np.min(values),
            'q25': np.percentile(values, 25), 
            'q75': np.percentile(values, 75), 
            'num_frames': len(values) 
        }
        
        statistics.append(stats)
    
    stats_df = pd.DataFrame(statistics)
    
    # Номера строк
    column_order = ['pair', 'min', 'max', 'range', 'mean', 'median', 'std', 'q25', 'q75', 'num_frames']
    stats_df = stats_df[column_order]
    stats_df.to_csv(output_csv_path, index=False)
    
    return stats_df

In [14]:
def calculate_statistics(input_csv_path, distance_prefix='dist_'):
    """Агрегирует попарные расстояния по кадрам.

    Новые признаки (итерация 7):
      velocity  — скорость изменения расстояния (первая производная)
      acceleration — ускорение (вторая производная)
      angle     — угол вектора i→j в плоскости XY (радианы)
    Перед дифференцированием применяется сглаживание Savitzky-Golay
    (window=5, polyorder=2) для подавления шума MediaPipe.
    """
    df = pd.read_csv(input_csv_path)
    print(f"Обрабатываю : {input_csv_path}")

    if '_with_eyes.csv' in input_csv_path:
        input_csv_path = input_csv_path.replace('_with_eyes.csv', '')
    output_csv_path = input_csv_path.replace('face_landmarks', 'statistics')

    distance_cols = [col for col in df.columns if col.startswith(distance_prefix)]
    if not distance_cols:
        raise ValueError(f"Не найдено столбцов с префиксом '{distance_prefix}'")
    print(f"Найдено {len(distance_cols)} столбцов с расстояниями")

    statistics = []

    for col in distance_cols:
        values = df[col].values.astype(float)
        n = len(values)

        # ── Сглаживание для производных ──────────────────────────────
        if n >= 5:
            win = 5  # нечётное, >= polyorder+2=4
            values_smooth = savgol_filter(values, window_length=win, polyorder=2)
        else:
            values_smooth = values.copy()

        velocity = np.diff(values_smooth)
        acceleration = np.diff(velocity) if len(velocity) > 1 else np.array([0.0])

        # ── Угол вектора i→j в плоскости XY ─────────────────────────
        try:
            parts = col.replace('dist_', '').split('_')
            i_pt, j_pt = int(parts[0]), int(parts[1])
            dx = df[f'x_{j_pt}'].values - df[f'x_{i_pt}'].values
            dy = df[f'y_{j_pt}'].values - df[f'y_{i_pt}'].values
            angle = np.arctan2(dy, dx)
            mean_angle  = float(np.mean(angle))
            std_angle   = float(np.std(angle))
            range_angle = float(np.ptp(angle))        # max - min
        except (KeyError, ValueError, IndexError):
            mean_angle = std_angle = range_angle = float('nan')

        stats = {
            'pair':         col,
            # ── существующие агрегаты ─────────────────────────────
            'min':          float(np.min(values)),
            'max':          float(np.max(values)),
            'std':          float(np.std(values)),
            'mean':         float(np.mean(values)),
            'median':       float(np.median(values)),
            'range':        float(np.ptp(values)),
            'q25':          float(np.percentile(values, 25)),
            'q75':          float(np.percentile(values, 75)),
            'num_frames':   n,
            # ── скорость (первая производная) ────────────────────
            'max_vel':      float(np.max(np.abs(velocity))),
            'mean_vel':     float(np.mean(np.abs(velocity))),
            'std_vel':      float(np.std(velocity)),
            # ── ускорение (вторая производная) ───────────────────
            'max_acc':      float(np.max(np.abs(acceleration))),
            'std_acc':      float(np.std(acceleration)),
            # ── угол вектора ──────────────────────────────────────
            'mean_angle':   mean_angle,
            'std_angle':    std_angle,
            'range_angle':  range_angle,
        }
        statistics.append(stats)

    stats_df = pd.DataFrame(statistics)

    column_order = [
        'pair', 'min', 'max', 'range', 'mean', 'median', 'std', 'q25', 'q75',
        'num_frames',
        'max_vel', 'mean_vel', 'std_vel',
        'max_acc', 'std_acc',
        'mean_angle', 'std_angle', 'range_angle',
    ]
    stats_df = stats_df[column_order]
    stats_df.to_csv(output_csv_path, index=False)

    return stats_df


In [15]:
def calculate_mean(input_csv_path, distance_prefix='dist_'):

    df = pd.read_csv(input_csv_path)
    print(f"Обрабатываю : {input_csv_path}")

    # if '_with_eyes.csv' in input_csv_path:
    #     input_csv_path = input_csv_path.replace('_with_eyes.csv', '')
    output_csv_path = input_csv_path.replace('face_landmarks', 'statistics')
    
    distance_cols = [col for col in df.columns if col.startswith(distance_prefix)]
    
    if not distance_cols:
        raise ValueError(f"Не найдено столбцов с префиксом '{distance_prefix}'")
    
    print(f"Найдено {len(distance_cols)} столбцов с расстояниями")
    
    statistics = []
    
    for col in distance_cols:
        values = df[col].values
        
        # статистики
        stats = {
            'pair': col,
            'mean': np.mean(values),
            'median': np.median(values), #возможно это информативнее
        }
        
        statistics.append(stats)
    
    stats_df = pd.DataFrame(statistics)
    
    # Номера строк
    column_order = ['pair', 'mean', 'median']
    stats_df = stats_df[column_order]
    stats_df.to_csv(output_csv_path, index=False)
    
    return stats_df


In [21]:
def normalize1(input_csv_path, norm_csv_path):

    df1 = pd.read_csv(input_csv_path)
    print(f"Обрабатываю : {input_csv_path}")
    df2 = pd.read_csv(norm_csv_path)
    
    # Объединяем и вычисляем
    merged = pd.merge(df1[['pair', 'min', 'max', 'std', 'mean']], 
                     df2[['pair', 'mean']].rename(columns={'mean': 'ref_mean'}),
                     on='pair')
    
    merged['min_norm'] = merged['min'] / merged['ref_mean']
    merged['max_norm'] = merged['max'] / merged['ref_mean']
    merged['std_norm'] = merged['std'] / merged['ref_mean']
    merged['mean_norm'] = merged['mean'] / merged['ref_mean']

    return merged[['pair', 'min_norm', 'max_norm', 'std_norm','mean_norm']]

In [16]:
def normalize(df1, df2, df_out):
    """Нормирует статистики упражнения на neutral-face ref_mean.

    Совместимость: базовые колонки (min_norm … std_norm) не меняются.
    Новые признаки:
      max_vel_norm, mean_vel_norm, std_vel_norm   — скорость / ref_mean
      max_acc_norm, std_acc_norm                  — ускорение / ref_mean
      mean_angle, std_angle, range_angle          — углы (рад, без норм.)
    Если новых колонок в df1 нет (старые файлы статистик), функция
    работает как прежде — обратная совместимость сохранена.
    """
    VEL_ACC_COLS = ['max_vel', 'mean_vel', 'std_vel', 'max_acc', 'std_acc']
    ANGLE_COLS   = ['mean_angle', 'std_angle', 'range_angle']

    # Берём только колонки, которые реально есть в df1
    extra_vel = [c for c in VEL_ACC_COLS if c in df1.columns]
    extra_ang = [c for c in ANGLE_COLS   if c in df1.columns]

    base = ['pair', 'min', 'max', 'mean', 'median', 'std', 'num_frames']
    use  = base + extra_vel + extra_ang

    merged = pd.merge(
        df1[[c for c in use if c in df1.columns]],
        df2[['pair', 'median']].rename(columns={'median': 'ref_mean'}),
        on='pair'
    )

    # ── базовые нормировки (как было) ────────────────────────────
    merged['min_norm']    = merged['min']    / merged['ref_mean']
    merged['max_norm']    = merged['max']    / merged['ref_mean']
    merged['mean_norm']   = merged['mean']   / merged['ref_mean']
    merged['median_norm'] = merged['median'] / merged['ref_mean']
    merged['std_norm']    = merged['std']    / merged['ref_mean']

    out_cols = ['pair', 'min_norm', 'max_norm', 'mean_norm',
                'median_norm', 'std_norm', 'num_frames']
    out = merged[out_cols].copy()

    # ── динамические признаки (нормируются на ref_mean) ──────────
    for col in extra_vel:
        out[f'{col}_norm'] = merged[col] / merged['ref_mean']

    # ── угловые признаки (уже безразмерны — радианы) ─────────────
    for col in extra_ang:
        out[col] = merged[col]

    out.to_csv(df_out, index=False)
    return out

In [ ]:
#ПРОХОЖУСЬ ПО ВИДЕО

path = 'D:\\HEALTHY\\'
folders = os.listdir(path) #список всех файлов
folders = ['Healthy29']


for e in folders:
    path2 = os.path.join(path, str(e))
    f2 = os.listdir(path2)
    for r in f2:
        if r=='r0':
            VIDEO_PATH_p3 = os.path.join(path2, str(r) + '\\face\\p3_' + str(e) + '.mp4')
            VIDEO_PATH_OUT = os.path.join(path2, str(r) + '\\mediapipe\\'+ '\\Brows\\')
            OUTPUT_DIR = os.path.dirname(VIDEO_PATH_OUT)
            try:
                raw_csv_path = process_video_facemesh(VIDEO_PATH_p3, OUTPUT_DIR, OUTPUT_DIR)
            except Exception as e:
                print(f"[FATAL] Не удалось извлечь данные Mediapipe: {e}") 

        if r=='r1':
            VIDEO_PATH_p3 = os.path.join(path2, str(r) + '\\face\\p3_m1_' + str(e) + '.mp4')
            VIDEO_PATH_OUT = os.path.join(path2, str(r) + '\\mediapipe\\'+ '\\Brows\\')
            OUTPUT_DIR = os.path.dirname(VIDEO_PATH_OUT)
            try:
                raw_csv_path = process_video_facemesh(VIDEO_PATH_p3, OUTPUT_DIR, OUTPUT_DIR)
            except Exception as e:
                print(f"[FATAL] Не удалось извлечь данные Mediapipe: {e}")
                
        elif r=='r2':
            VIDEO_PATH_p3 = os.path.join(path2, str(r) + '\\face\\p3_m2_' + str(e) + '.mp4')
            VIDEO_PATH_OUT = os.path.join(path2, str(r) + '\\mediapipe\\'+ '\\Brows\\')
            OUTPUT_DIR = os.path.dirname(VIDEO_PATH_OUT)
            try:
                raw_csv_path = process_video_facemesh(VIDEO_PATH_p3, OUTPUT_DIR, OUTPUT_DIR)
            except Exception as e:
                print(f"[FATAL] Не удалось извлечь данные Mediapipe: {e}")
                
        elif r=='r3':
            VIDEO_PATH_p3 = os.path.join(path2, str(r) + '\\face\\p3_m3_' + str(e) + '.mp4')
            VIDEO_PATH_OUT = os.path.join(path2, str(r) + '\\mediapipe\\'+ '\\Brows\\')
            OUTPUT_DIR = os.path.dirname(VIDEO_PATH_OUT)
            try:
                raw_csv_path = process_video_facemesh(VIDEO_PATH_p3, OUTPUT_DIR, OUTPUT_DIR)
            except Exception as e:
                print(f"[FATAL] Не удалось извлечь данные Mediapipe: {e}")

        elif r=='r4':
            VIDEO_PATH_p3 = os.path.join(path2, str(r) + '\\face\\p3_m4_' + str(e) + '.mp4')
            VIDEO_PATH_OUT = os.path.join(path2, str(r) + '\\mediapipe\\'+ '\\Brows\\')
            OUTPUT_DIR = os.path.dirname(VIDEO_PATH_OUT)
            try:
                raw_csv_path = process_video_facemesh(VIDEO_PATH_p3, OUTPUT_DIR, OUTPUT_DIR)
            except Exception as e:
                print(f"[FATAL] Не удалось извлечь данные Mediapipe: {e}")

                
        elif r=='r5':
            VIDEO_PATH_p3 = os.path.join(path2, str(r) + '\\face\\p3_m5_' + str(e) + '.mp4')
            VIDEO_PATH_OUT = os.path.join(path2, str(r) + '\\mediapipe\\'+ '\\Brows\\')
            OUTPUT_DIR = os.path.dirname(VIDEO_PATH_OUT)
            try:
                raw_csv_path = process_video_facemesh(VIDEO_PATH_p3, OUTPUT_DIR, OUTPUT_DIR)
            except Exception as e:
                print(f"[FATAL] Не удалось извлечь данные Mediapipe: {e}")


In [40]:
#ПРЕОБРАЗОВАНИЕ CSV

path = 'D:\\HEALTHY\\'
folders = os.listdir(path) #список всех файлов
folders = ['Healthy29']


for e in folders:
    path2 = os.path.join(path, str(e))
    f2 = os.listdir(path2)
    for r in f2:
        if r=='r0':
            CSV_PATH_p3 = os.path.join(path2, str(r) + '\\mediapipe\\Brows\\p3_' + str(e) + '_face_landmarks'+'.csv')
            try:
                tr_csv = transform_landmarks_csv(CSV_PATH_p3)
            except Exception as e:
                print(f"[FATAL] Не удалось преобразовать таблицу: {e}") 

        elif r=='r1':
            CSV_PATH_p3 = os.path.join(path2, str(r) + '\\mediapipe\\Brows\\p3_m1_' + str(e) + '_face_landmarks'+'.csv')
            try:
                tr_csv = transform_landmarks_csv(CSV_PATH_p3)
            except Exception as e:
                print(f"[FATAL] Не удалось преобразовать таблицу: {e}") 
                
        elif r=='r2':
            CSV_PATH_p3 = os.path.join(path2, str(r) + '\\mediapipe\\Brows\\p3_m2_' + str(e) + '_face_landmarks'+'.csv')
            try:
                tr_csv = transform_landmarks_csv(CSV_PATH_p3)
            except Exception as e:
                print(f"[FATAL] Не удалось преобразовать таблицу: {e}")
                
        elif r=='r3':
            CSV_PATH_p3 = os.path.join(path2, str(r) + '\\mediapipe\\Brows\\p3_m3_' + str(e) + '_face_landmarks'+'.csv')
            try:
                tr_csv = transform_landmarks_csv(CSV_PATH_p3)
            except Exception as e:
                print(f"[FATAL] Не удалось преобразовать таблицу: {e}")

        elif r=='r4':
            CSV_PATH_p3 = os.path.join(path2, str(r) + '\\mediapipe\\Brows\\p3_m4_' + str(e) + '_face_landmarks'+'.csv')
            try:
                tr_csv = transform_landmarks_csv(CSV_PATH_p3)
            except Exception as e:
                print(f"[FATAL] Не удалось преобразовать таблицу: {e}")

                
        elif r=='r5':
            CSV_PATH_p3 = os.path.join(path2, str(r) + '\\mediapipe\\Brows\\p3_m5_' + str(e) + '_face_landmarks'+'.csv')
            try:
                tr_csv = transform_landmarks_csv(CSV_PATH_p3)
            except Exception as e:
                print(f"[FATAL] Не удалось преобразовать таблицу: {e}")

Обработка файла: D:\HEALTHY\Healthy29\r0\mediapipe\Brows\p3_Healthy29_face_landmarks.csv
Исходный размер: (190244, 6)
Преобразование завершено!
Новая таблица имеет размер: (398, 1436)
Количество фреймов: 398
Количество колонок: 1436
Сохранено в: D:\HEALTHY\Healthy29\r0\mediapipe\Brows\p3_Healthy29_face_landmarks.csv
Обработка файла: D:\HEALTHY\Healthy29\r1\mediapipe\Brows\p3_m1_Healthy29_face_landmarks.csv
Исходный размер: (178772, 6)
Преобразование завершено!
Новая таблица имеет размер: (374, 1436)
Количество фреймов: 374
Количество колонок: 1436
Сохранено в: D:\HEALTHY\Healthy29\r1\mediapipe\Brows\p3_m1_Healthy29_face_landmarks.csv


In [ ]:
path = 'D:\\HEALTHY\\'
folders = os.listdir(path) #список всех файлов
# folders = ['Healthy29']


for e in folders:
    path2 = os.path.join(path, str(e))
    f2 = os.listdir(path2)
    for r in f2:
        if r=='r0':
            CSV_PATH_p3 = os.path.join(path2, str(r) + '\\mediapipe\\NEUTRAL\\p1_' + str(e) + '_face_landmarks_with_EAR'+'.csv')
            if os.path.exists(CSV_PATH_p3):
                try:
                    distances = dist_beetw_eyes(CSV_PATH_p3)
                except Exception as e:
                    print(f"[FATAL] Не удалось преобразовать таблицу: {e}")
            else:
                print(f"[WARNING] Файл не найден: {CSV_PATH_p3}")

        elif r=='r1':
            CSV_PATH_p3 = os.path.join(path2, str(r) + '\\mediapipe\\NEUTRAL\\p1_m1_' + str(e) + '_face_landmarks_with_EAR'+'.csv')
            if os.path.exists(CSV_PATH_p3):
                try:
                    distances = dist_beetw_eyes(CSV_PATH_p3)
                except Exception as e:
                    print(f"[FATAL] Не удалось преобразовать таблицу: {e}")
            else:
                print(f"[WARNING] Файл не найден: {CSV_PATH_p3}")
                
        elif r=='r2':
            CSV_PATH_p3 = os.path.join(path2, str(r) + '\\mediapipe\\NEUTRAL\\p1_m2_' + str(e) + '_face_landmarks_with_EAR'+'.csv')
            if os.path.exists(CSV_PATH_p3):
                try:
                    distances = dist_beetw_eyes(CSV_PATH_p3)
                except Exception as e:
                    print(f"[FATAL] Не удалось преобразовать таблицу: {e}")
            else:
                print(f"[WARNING] Файл не найден: {CSV_PATH_p3}")
                
        elif r=='r3':
            CSV_PATH_p3 = os.path.join(path2, str(r) + '\\mediapipe\\NEUTRAL\\p1_m3_' + str(e) + '_face_landmarks_with_EAR'+'.csv')
            if os.path.exists(CSV_PATH_p3):
                try:
                    distances = dist_beetw_eyes(CSV_PATH_p3)
                except Exception as e:
                    print(f"[FATAL] Не удалось преобразовать таблицу: {e}")
            else:
                print(f"[WARNING] Файл не найден: {CSV_PATH_p3}")
                
        elif r=='r4':
            CSV_PATH_p3 = os.path.join(path2, str(r) + '\\mediapipe\\NEUTRAL\\p1_m4_' + str(e) + '_face_landmarks_with_EAR'+'.csv')
            if os.path.exists(CSV_PATH_p3):
                try:
                    distances = dist_beetw_eyes(CSV_PATH_p3)
                except Exception as e:
                    print(f"[FATAL] Не удалось преобразовать таблицу: {e}")
            else:
                print(f"[WARNING] Файл не найден: {CSV_PATH_p3}")
                
        elif r=='r5':
            CSV_PATH_p3 = os.path.join(path2, str(r) + '\\mediapipe\\NEUTRAL\\p1_m5_' + str(e) + '_face_landmarks_with_EAR'+'.csv')
            if os.path.exists(CSV_PATH_p3):
                try:
                    distances = dist_beetw_eyes(CSV_PATH_p3)
                except Exception as e:
                    print(f"[FATAL] Не удалось преобразовать таблицу: {e}")
            else:
                print(f"[WARNING] Файл не найден: {CSV_PATH_p3}")

In [52]:
import os
import shutil
import pandas as pd


def get_healthy_brows_filename(patient_name, r_num):
    """
    r0 -> p3_HealthyX_face_landmarks.csv
    r1 -> p3_m1_HealthyX_face_landmarks.csv
    r2 -> p3_m2_HealthyX_face_landmarks.csv
    ...
    r5 -> p3_m5_HealthyX_face_landmarks.csv
    """

    if r_num == 0:
        return f'p3_{patient_name}_face_landmarks.csv'
    else:
        return f'p3_m{r_num}_{patient_name}_face_landmarks.csv'


def create_brows2_without_dist_columns(
    root_dir=r'D:\HEALTHY',
    patient_start=1,
    patient_end=133,
    r_values=range(0, 6)
):
    processed_files = []
    skipped_files = []
    errors = []

    for patient_num in range(patient_start, patient_end + 1):

        patient_name = f'Healthy{patient_num}'

        for r_num in r_values:

            brows_dir = os.path.join(
                root_dir,
                patient_name,
                f'r{r_num}',
                'mediapipe',
                'Brows'
            )

            brows2_dir = os.path.join(
                root_dir,
                patient_name,
                f'r{r_num}',
                'mediapipe',
                'Brows2'
            )

            filename = get_healthy_brows_filename(
                patient_name,
                r_num
            )

            input_csv = os.path.join(brows_dir, filename)
            output_csv = os.path.join(brows2_dir, filename)

            if not os.path.exists(input_csv):
                skipped_files.append(input_csv)
                continue

            try:
                os.makedirs(brows2_dir, exist_ok=True)

                shutil.copy2(input_csv, output_csv)

                df = pd.read_csv(output_csv)

                dist_columns = [
                    col for col in df.columns
                    if col.startswith('dist_')
                    and col != 'distance_eyes'
                ]

                df = df.drop(columns=dist_columns)

                df.to_csv(output_csv, index=False)

                print(f'Обработан: {output_csv}')
                print(f'Удалено dist колонок: {len(dist_columns)}')

                processed_files.append(output_csv)

            except Exception as e:
                print(f'Ошибка: {input_csv}')
                print(e)
                errors.append((input_csv, str(e)))

    print('\n=====================================')
    print(f'Обработано файлов: {len(processed_files)}')
    print(f'Пропущено файлов: {len(skipped_files)}')
    print(f'Ошибок: {len(errors)}')
    print('=====================================')

    return processed_files, skipped_files, errors


processed_files, skipped_files, errors = create_brows2_without_dist_columns(
    root_dir=r'D:\HEALTHY',
    patient_start=1,
    patient_end=133,
    r_values=range(0, 6)
)

Обработан: D:\HEALTHY\Healthy1\r0\mediapipe\Brows2\p3_Healthy1_face_landmarks.csv
Удалено dist колонок: 24
Обработан: D:\HEALTHY\Healthy2\r0\mediapipe\Brows2\p3_Healthy2_face_landmarks.csv
Удалено dist колонок: 24
Обработан: D:\HEALTHY\Healthy2\r1\mediapipe\Brows2\p3_m1_Healthy2_face_landmarks.csv
Удалено dist колонок: 24
Обработан: D:\HEALTHY\Healthy2\r2\mediapipe\Brows2\p3_m2_Healthy2_face_landmarks.csv
Удалено dist колонок: 24
Обработан: D:\HEALTHY\Healthy2\r3\mediapipe\Brows2\p3_m3_Healthy2_face_landmarks.csv
Удалено dist колонок: 24
Обработан: D:\HEALTHY\Healthy2\r4\mediapipe\Brows2\p3_m4_Healthy2_face_landmarks.csv
Удалено dist колонок: 24
Обработан: D:\HEALTHY\Healthy2\r5\mediapipe\Brows2\p3_m5_Healthy2_face_landmarks.csv
Удалено dist колонок: 24
Обработан: D:\HEALTHY\Healthy3\r0\mediapipe\Brows2\p3_Healthy3_face_landmarks.csv
Удалено dist колонок: 24
Обработан: D:\HEALTHY\Healthy3\r1\mediapipe\Brows2\p3_m1_Healthy3_face_landmarks.csv
Удалено dist колонок: 24
Обработан: D:\HEALT

In [ ]:
path = 'D:\\HEALTHY\\'
folders = os.listdir(path) #список всех файлов
# folders = ['Patient1']



for e in folders:
    path2 = os.path.join(path, str(e))
    f2 = os.listdir(path2)
    for r in f2:
        if r=='r0':
            CSV_PATH_p3 = os.path.join(path2, str(r) + '\\mediapipe\\Brows2\\p3_' + str(e) + '_face_landmarks'+'.csv')
            if os.path.exists(CSV_PATH_p3):
                try:
                    df1 = pd.read_csv(CSV_PATH_p3)
                    print(f"Исходный датафрейм: {df1.shape}")
                    df_full = calculate_distances_flexible(CSV_PATH_p3)
                    print(f"Полный датафрейм: {df_full.shape}")
                except Exception as e:
                    print(f"[FATAL] Не удалось преобразовать таблицу: {e}")
            else:
                print(f"[WARNING] Файл не найден: {CSV_PATH_p3}")
            

        elif r=='r1':
            CSV_PATH_p3 = os.path.join(path2, str(r) + '\\mediapipe\\Brows2\\p3_m1_' + str(e) + '_face_landmarks'+'.csv')
            if os.path.exists(CSV_PATH_p3):
                try:
                    df1 = pd.read_csv(CSV_PATH_p3)
                    print(f"Исходный датафрейм: {df1.shape}")
                    df_full = calculate_distances_flexible(CSV_PATH_p3)
                    print(f"Полный датафрейм: {df_full.shape}")
                except Exception as e:
                    print(f"[FATAL] Не удалось преобразовать таблицу: {e}")
            else:
                print(f"[WARNING] Файл не найден: {CSV_PATH_p3}")
                
        elif r=='r2':
            CSV_PATH_p3 = os.path.join(path2, str(r) + '\\mediapipe\\Brows2\\p3_m2_' + str(e) + '_face_landmarks'+'.csv')
            if os.path.exists(CSV_PATH_p3):
                try:
                    df1 = pd.read_csv(CSV_PATH_p3)
                    print(f"Исходный датафрейм: {df1.shape}")
                    df_full = calculate_distances_flexible(CSV_PATH_p3)
                    print(f"Полный датафрейм: {df_full.shape}")
                except Exception as e:
                    print(f"[FATAL] Не удалось преобразовать таблицу: {e}")
                print(f"[WARNING] Файл не найден: {CSV_PATH_p3}")
                
        elif r=='r3':
            CSV_PATH_p3 = os.path.join(path2, str(r) + '\\mediapipe\\Brows2\\p3_m3_' + str(e) + '_face_landmarks'+'.csv')
            if os.path.exists(CSV_PATH_p3):
                try:
                    df1 = pd.read_csv(CSV_PATH_p3)
                    print(f"Исходный датафрейм: {df1.shape}")
                    df_full = calculate_distances_flexible(CSV_PATH_p3)
                    print(f"Полный датафрейм: {df_full.shape}")
                except Exception as e:
                    print(f"[FATAL] Не удалось преобразовать таблицу: {e}")
            else:
                print(f"[WARNING] Файл не найден: {CSV_PATH_p3}")

        elif r=='r4':
            CSV_PATH_p3 = os.path.join(path2, str(r) + '\\mediapipe\\Brows2\\p3_m4_' + str(e) + '_face_landmarks'+'.csv')
            if os.path.exists(CSV_PATH_p3):
                try:
                    df1 = pd.read_csv(CSV_PATH_p3)
                    print(f"Исходный датафрейм: {df1.shape}")
                    df_full = calculate_distances_flexible(CSV_PATH_p3)
                    print(f"Полный датафрейм: {df_full.shape}")
                except Exception as e:
                    print(f"[FATAL] Не удалось преобразовать таблицу: {e}")
            else:
                print(f"[WARNING] Файл не найден: {CSV_PATH_p3}")

                
        elif r=='r5':
            CSV_PATH_p3 = os.path.join(path2, str(r) + '\\mediapipe\\Brows2\\p3_m5_' + str(e) + '_face_landmarks'+'.csv')
            if os.path.exists(CSV_PATH_p3):
                try:
                    df1 = pd.read_csv(CSV_PATH_p3)
                    print(f"Исходный датафрейм: {df1.shape}")
                    df_full = calculate_distances_flexible(CSV_PATH_p3)
                    print(f"Полный датафрейм: {df_full.shape}")
                except Exception as e:
                    print(f"[FATAL] Не удалось преобразовать таблицу: {e}")
            else:
                print(f"[WARNING] Файл не найден: {CSV_PATH_p3}")

In [49]:
# ОБРЕЗКА КАДРОВ ДЛЯ PD ПАЦИЕНТОВ
# Читает start/end из PD_num_frames_norm.csv и обрезает face_landmarks CSV по колонке frame_num

import pandas as pd
import os

# Путь к файлу с метками кадров — поправь, если нужно
FRAMES_CSV = r"C:\Users\Ekaterina\Desktop\PD_num_frames_norm_brows.csv"

# Загружаем таблицу с диапазонами кадров
frames_df = pd.read_csv(FRAMES_CSV, sep=';', encoding='cp1251')
frames_df = frames_df[['п»їp', 'r', 'start', 'end']].dropna(subset=['п»їp', 'r', 'start', 'end'])
frames_df['п»їp'] = frames_df['п»їp'].astype(int)
frames_df['r']       = frames_df['r'].astype(int)
frames_df['start']   = frames_df['start'].astype(int)
frames_df['end']     = frames_df['end'].astype(int)

path = 'D:\\PD\\'
folders = os.listdir(path)

for e in folders:
    # Извлекаем номер пациента из имени папки (Patient2 -> 2)
    try:
        patient_num = int(str(e).replace('Patient', '').strip())
    except ValueError:
        print(f"[WARNING] Не удалось определить номер пациента: {e}")
        continue

    path2 = os.path.join(path, str(e))
    if not os.path.isdir(path2):
        continue

    for r in os.listdir(path2):
        if r not in ['r0', 'r1','r1','r2','r3','r4', 'r5']:
            continue

        r_num = int(r[1])  # 'r0' -> 0, 'r1' -> 1

        # Ищем строку с нужным пациентом и попыткой
        row = frames_df[(frames_df['п»їp'] == patient_num) & (frames_df['r'] == r_num)]
        if row.empty:
            print(f"[WARNING] Нет данных о кадрах для {e}, {r}")
            continue

        start_frame = row.iloc[0]['start']
        end_frame   = row.iloc[0]['end']

        # Формируем пути к CSV файлам
        if r == 'r0':
            # CSV_PATH_p4 = os.path.join(path2, r, 'mediapipe', 'Smile',
            #     f'p4_{e}_face_landmarks.csv_with_eyes.csv')
            CSV_PATH_p3 = os.path.join(path2, r, 'mediapipe', 'Brows',
                f'p3_{e}_face_landmarks.csv')
        else:
            m = r[1]  # номер попытки
            # CSV_PATH_p4 = os.path.join(path2, r, 'mediapipe', 'Smile',
            #     f'p4_{e}_face_landmarks.csv_with_eyes.csv')
            CSV_PATH_p3 = os.path.join(path2, r, 'mediapipe', 'Brows',
                f'p3_{e}_face_landmarks.csv')

        for csv_path in [CSV_PATH_p3]:
            if not os.path.exists(csv_path):
                print(f"[WARNING] Файл не найден: {csv_path}")
                continue
            try:
                df = pd.read_csv(csv_path)
                before = len(df)
                df_trimmed = df[(df['frame_number'] >= start_frame) & (df['frame_number'] <= end_frame)].copy()
                df_trimmed.to_csv(csv_path, index=False)
                print(f"[OK] {os.path.basename(csv_path)}: {before} -> {len(df_trimmed)} кадров "
                      f"(start={start_frame}, end={end_frame})")
            except Exception as ex:
                print(f"[FATAL] Ошибка при обработке {csv_path}: {ex}")


[OK] p3_Patient1_face_landmarks.csv: 968 -> 700 кадров (start=228, end=927)
[OK] p3_Patient10_face_landmarks.csv: 798 -> 651 кадров (start=92, end=742)
[OK] p3_Patient100_face_landmarks.csv: 714 -> 641 кадров (start=46, end=686)
[OK] p3_Patient100_face_landmarks.csv: 636 -> 604 кадров (start=32, end=635)
[OK] p3_Patient102_face_landmarks.csv: 316 -> 268 кадров (start=31, end=298)
[OK] p3_Patient103_face_landmarks.csv: 1115 -> 941 кадров (start=13, end=953)
[OK] p3_Patient103_face_landmarks.csv: 859 -> 787 кадров (start=17, end=803)
[OK] p3_Patient104_face_landmarks.csv: 1169 -> 871 кадров (start=92, end=962)
[OK] p3_Patient104_face_landmarks.csv: 897 -> 802 кадров (start=23, end=824)
[OK] p3_Patient105_face_landmarks.csv: 927 -> 784 кадров (start=95, end=878)
[OK] p3_Patient105_face_landmarks.csv: 921 -> 810 кадров (start=86, end=895)
[OK] p3_Patient106_face_landmarks.csv: 483 -> 452 кадров (start=10, end=461)
[OK] p3_Patient106_face_landmarks.csv: 761 -> 691 кадров (start=32, end=722)

In [48]:
# ОБРЕЗКА КАДРОВ ДЛЯ PD ПАЦИЕНТОВ
# Читает start/end из PD_num_frames_norm.csv и обрезает face_landmarks CSV по колонке frame_num

import pandas as pd
import os

# Путь к файлу с метками кадров — поправь, если нужно
FRAMES_CSV = r"C:\Users\Ekaterina\Desktop\PD_num_frames_norm_brows.csv"

# Загружаем таблицу с диапазонами кадров
frames_df = pd.read_csv(FRAMES_CSV, sep=';', encoding='cp1251')

# ДИАГНОСТИКА: выводим все колонки и первые строки
print("Доступные колонки в файле:", frames_df.columns.tolist())
print("\nПервые 5 строк:")
print(frames_df.head())

# Проверяем, какие колонки из нужных есть
needed_cols = ['patient', 'r', 'start', 'end']
available_cols = [col for col in needed_cols if col in frames_df.columns]
missing_cols = [col for col in needed_cols if col not in frames_df.columns]

print(f"\nПрисутствующие колонки: {available_cols}")
print(f"Отсутствующие колонки: {missing_cols}")

# Если нет колонки 'patient', возможно она называется иначе, например:
# 'Patient', 'patient_id', 'id', 'subj', 'subject' и т.д.

# Если нашли похожую колонку, используйте её имя
# Например:
# patient_col = 'Patient'  # если в CSV колонка называется 'Patient'

Доступные колонки в файле: ['п»їp', 'r', 'start', 'end', 'num_frames_norm']

Первые 5 строк:
   п»їp  r  start  end  num_frames_norm
0     1  0    228  927              699
1     2  0     80  526              446
2     3  0      6  515              509
3     4  0     31  477              446
4     5  0     44  457              413

Присутствующие колонки: ['r', 'start', 'end']
Отсутствующие колонки: ['patient']


In [17]:
path = 'D:\\PD\\'
folders = os.listdir(path) #список всех файлов
# folders = ['Patient92']


for e in folders:
    path2 = os.path.join(path, str(e))
    f2 = os.listdir(path2)
    for r in f2:
        if r=='r0':
            CSV_PATH_p3 = os.path.join(path2, str(r) + '\\mediapipe\\Brows2\\p3_' + str(e) + '_face_landmarks'+'.csv')
            if os.path.exists(CSV_PATH_p3):
                try:
                    stats_df = calculate_statistics(CSV_PATH_p3)
                except Exception as e:
                    print(f"[FATAL] Не удалось преобразовать таблицу: {e}")
            else:
                print(f"[WARNING] Файл не найден: {CSV_PATH_p3}")

        elif r=='r1':
            CSV_PATH_p3 = os.path.join(path2, str(r) + '\\mediapipe\\Brows2\\p3_' + str(e) + '_face_landmarks'+'.csv')
            if os.path.exists(CSV_PATH_p3):
                try:
                    stats_df = calculate_statistics(CSV_PATH_p3)
                except Exception as e:
                    print(f"[FATAL] Не удалось преобразовать таблицу: {e}")
            else:
                print(f"[WARNING] Файл не найден: {CSV_PATH_p3}")
                
        elif r=='r2':
            CSV_PATH_p3 = os.path.join(path2, str(r) + '\\mediapipe\\Brows2\\p3_' + str(e) + '_face_landmarks'+'.csv')
            if os.path.exists(CSV_PATH_p3):
                try:
                    stats_df = calculate_statistics(CSV_PATH_p3)
                except Exception as e:
                    print(f"[FATAL] Не удалось преобразовать таблицу: {e}")
            else:
                print(f"[WARNING] Файл не найден: {CSV_PATH_p3}") 
                
        # elif r=='r3':
        #     CSV_PATH_p3 = os.path.join(path2, str(r) + '\\mediapipe\\Brows2\\p3_m3_' + str(e) + '_face_landmarks'+'.csv')
        #     if os.path.exists(CSV_PATH_p3):
        #         try:
        #             stats_df = calculate_statistics(CSV_PATH_p3)
        #         except Exception as e:
        #             print(f"[FATAL] Не удалось преобразовать таблицу: {e}")
        #     else:
        #         print(f"[WARNING] Файл не найден: {CSV_PATH_p3}")

        # elif r=='r4':
        #     CSV_PATH_p3 = os.path.join(path2, str(r) + '\\mediapipe\\Brows2\\p3_m4_' + str(e) + '_face_landmarks'+'.csv')
        #     if os.path.exists(CSV_PATH_p3):
        #         try:
        #             stats_df = calculate_statistics(CSV_PATH_p3)
        #         except Exception as e:
        #             print(f"[FATAL] Не удалось преобразовать таблицу: {e}")
        #     else:
        #         print(f"[WARNING] Файл не найден: {CSV_PATH_p3}")
                
        # elif r=='r5':
        #     CSV_PATH_p3 = os.path.join(path2, str(r) + '\\mediapipe\\Brows2\\p3_m5_' + str(e) + '_face_landmarks'+'.csv')
        #     if os.path.exists(CSV_PATH_p3):
        #         try:
        #             stats_df = calculate_statistics(CSV_PATH_p3)
        #         except Exception as e:
        #             print(f"[FATAL] Не удалось преобразовать таблицу: {e}")
        #     else:
        #         print(f"[WARNING] Файл не найден: {CSV_PATH_p3}")

Обрабатываю : D:\PD\Patient1\r0\mediapipe\Brows2\p3_Patient1_face_landmarks.csv
Найдено 24 столбцов с расстояниями
Обрабатываю : D:\PD\Patient10\r0\mediapipe\Brows2\p3_Patient10_face_landmarks.csv
Найдено 24 столбцов с расстояниями
Обрабатываю : D:\PD\Patient100\r0\mediapipe\Brows2\p3_Patient100_face_landmarks.csv
Найдено 24 столбцов с расстояниями
Обрабатываю : D:\PD\Patient100\r1\mediapipe\Brows2\p3_Patient100_face_landmarks.csv
Найдено 24 столбцов с расстояниями
Обрабатываю : D:\PD\Patient102\r0\mediapipe\Brows2\p3_Patient102_face_landmarks.csv
Найдено 24 столбцов с расстояниями
Обрабатываю : D:\PD\Patient103\r0\mediapipe\Brows2\p3_Patient103_face_landmarks.csv
Найдено 24 столбцов с расстояниями
Обрабатываю : D:\PD\Patient103\r1\mediapipe\Brows2\p3_Patient103_face_landmarks.csv
Найдено 24 столбцов с расстояниями
Обрабатываю : D:\PD\Patient104\r0\mediapipe\Brows2\p3_Patient104_face_landmarks.csv
Найдено 24 столбцов с расстояниями
Обрабатываю : D:\PD\Patient104\r1\mediapipe\Brows2\p3_

In [19]:
import os
import shutil
import pandas as pd


BASE_PATH = r'D:\PD'


for patient_folder in os.listdir(BASE_PATH):

    if not patient_folder.startswith('Patient'):
        continue

    patient_path = os.path.join(BASE_PATH, patient_folder)

    if not os.path.isdir(patient_path):
        continue

    for r_num in range(0, 6):

        r_folder = f'r{r_num}'

        neutral_path = os.path.join(
            patient_path,
            r_folder,
            'mediapipe',
            'NEUTRAL'
        )

        fr_path = os.path.join(
            neutral_path,
        )

        if not os.path.isdir(fr_path):
            print(f'[WARNING] Нет папки fr: {fr_path}')
            continue

        # =====================================================
        # Для PD:
        # имя одинаковое для r0-r5
        # m НЕ добавляется
        # =====================================================

        source_filename = (
            f'p1_{patient_folder}_face_landmarks_with_EAR_blink_interpolated.csv'
        )

        source_path = os.path.join(
            fr_path,
            source_filename
        )

        if not os.path.exists(source_path):
            print(f'[WARNING] Файл не найден: {source_path}')
            continue

        # =====================================================
        # Новый итоговый файл
        # =====================================================

        output_filename = (
            f'p1_{patient_folder}_face_landmarks_with_EAR_final.csv'
        )

        output_path = os.path.join(
            fr_path,
            output_filename
        )

        # =====================================================
        # Копируем файл
        # =====================================================

        shutil.copy2(source_path, output_path)

        try:

            df = pd.read_csv(output_path)

            # =================================================
            # Удаляем только ОРИГИНАЛЬНЫЕ dist_*
            # Оставляем dist_*_clean
            # =================================================

            original_dist_cols = [
                col for col in df.columns
                if (
                    col.startswith('dist_')
                    and not col.endswith('_clean')
                )
            ]

            df = df.drop(columns=original_dist_cols)

            # =================================================
            # Переименовываем:
            # dist_XXX_clean -> dist_XXX
            # =================================================

            rename_dict = {}

            for col in df.columns:

                if col.endswith('_clean'):

                    new_name = col.replace('_clean', '')

                    rename_dict[col] = new_name

            df = df.rename(columns=rename_dict)

            # =================================================
            # Сохранение
            # =================================================

            df.to_csv(output_path, index=False)

            print(
                f'[OK] {output_filename}: '
                f'удалено dist колонок: {len(original_dist_cols)}, '
                f'переименовано clean колонок: {len(rename_dict)}'
            )

        except Exception as ex:

            print(f'[FATAL] Ошибка при обработке {output_path}: {ex}')

[OK] p1_Patient1_face_landmarks_with_EAR_final.csv: удалено dist колонок: 24, переименовано clean колонок: 24
[WARNING] Нет папки fr: D:\PD\Patient1\r1\mediapipe\NEUTRAL
[WARNING] Нет папки fr: D:\PD\Patient1\r2\mediapipe\NEUTRAL
[WARNING] Нет папки fr: D:\PD\Patient1\r3\mediapipe\NEUTRAL
[WARNING] Нет папки fr: D:\PD\Patient1\r4\mediapipe\NEUTRAL
[WARNING] Нет папки fr: D:\PD\Patient1\r5\mediapipe\NEUTRAL
[OK] p1_Patient10_face_landmarks_with_EAR_final.csv: удалено dist колонок: 24, переименовано clean колонок: 24
[WARNING] Нет папки fr: D:\PD\Patient10\r1\mediapipe\NEUTRAL
[WARNING] Нет папки fr: D:\PD\Patient10\r2\mediapipe\NEUTRAL
[WARNING] Нет папки fr: D:\PD\Patient10\r3\mediapipe\NEUTRAL
[WARNING] Нет папки fr: D:\PD\Patient10\r4\mediapipe\NEUTRAL
[WARNING] Нет папки fr: D:\PD\Patient10\r5\mediapipe\NEUTRAL
[OK] p1_Patient100_face_landmarks_with_EAR_final.csv: удалено dist колонок: 24, переименовано clean колонок: 24
[OK] p1_Patient100_face_landmarks_with_EAR_final.csv: удалено 

In [20]:
path = 'D:\\PD\\'
folders = os.listdir(path) #список всех файлов
# folders = ['Patient92']


for e in folders:
    path2 = os.path.join(path, str(e))
    f2 = os.listdir(path2)
    for r in f2:
        if r=='r0':
            CSV_PATH_p3 = os.path.join(path2, str(r) + '\\mediapipe\\NEUTRAL\\p1_' + str(e) + '_face_landmarks_with_EAR_final'+'.csv')
            if os.path.exists(CSV_PATH_p3):
                try:
                    stats_df = calculate_mean(CSV_PATH_p3)
                except Exception as e:
                    print(f"[FATAL] Не удалось преобразовать таблицу: {e}")
            else:
                print(f"[WARNING] Файл не найден: {CSV_PATH_p3}")

        elif r=='r1':
            CSV_PATH_p3 = os.path.join(path2, str(r) + '\\mediapipe\\NEUTRAL\\p1_' + str(e) + '_face_landmarks_with_EAR_final'+'.csv')
            if os.path.exists(CSV_PATH_p3):
                try:
                    stats_df = calculate_mean(CSV_PATH_p3)
                except Exception as e:
                    print(f"[FATAL] Не удалось преобразовать таблицу: {e}")
            else:
                print(f"[WARNING] Файл не найден: {CSV_PATH_p3}")
                
        elif r=='r2':
            CSV_PATH_p3 = os.path.join(path2, str(r) + '\\mediapipe\\NEUTRAL\\p1_' + str(e) + '_face_landmarks_with_EAR_final'+'.csv')
            if os.path.exists(CSV_PATH_p3):
                try:
                    stats_df = calculate_mean(CSV_PATH_p3)
                except Exception as e:
                    print(f"[FATAL] Не удалось преобразовать таблицу: {e}")
            else:
                print(f"[WARNING] Файл не найден: {CSV_PATH_p3}") 
                
        # elif r=='r3':
        #     CSV_PATH_p3 = os.path.join(path2, str(r) + '\\mediapipe\\NEUTRAL\\fr\\p1_m3_' + str(e) + '_face_landmarks_with_EAR'+'.csv')
        #     if os.path.exists(CSV_PATH_p3):
        #         try:
        #             stats_df = calculate_mean(CSV_PATH_p3)
        #         except Exception as e:
        #             print(f"[FATAL] Не удалось преобразовать таблицу: {e}")
        #     else:
        #         print(f"[WARNING] Файл не найден: {CSV_PATH_p3}")

        # elif r=='r4':
        #     CSV_PATH_p3 = os.path.join(path2, str(r) + '\\mediapipe\\NEUTRAL\\fr\\p1_m4_' + str(e) + '_face_landmarks_with_EAR'+'.csv')
        #     if os.path.exists(CSV_PATH_p3):
        #         try:
        #             stats_df = calculate_mean(CSV_PATH_p3)
        #         except Exception as e:
        #             print(f"[FATAL] Не удалось преобразовать таблицу: {e}")
        #     else:
        #         print(f"[WARNING] Файл не найден: {CSV_PATH_p3}")
                
        # elif r=='r5':
        #     CSV_PATH_p3 = os.path.join(path2, str(r) + '\\mediapipe\\NEUTRAL\\fr\\p1_m5_' + str(e) + '_face_landmarks_with_EAR'+'.csv')
        #     if os.path.exists(CSV_PATH_p3):
        #         try:
        #             stats_df = calculate_mean(CSV_PATH_p3)
        #         except Exception as e:
        #             print(f"[FATAL] Не удалось преобразовать таблицу: {e}")
        #     else:
        #         print(f"[WARNING] Файл не найден: {CSV_PATH_p3}")

Обрабатываю : D:\PD\Patient1\r0\mediapipe\NEUTRAL\p1_Patient1_face_landmarks_with_EAR_final.csv
Найдено 24 столбцов с расстояниями
Обрабатываю : D:\PD\Patient10\r0\mediapipe\NEUTRAL\p1_Patient10_face_landmarks_with_EAR_final.csv
Найдено 24 столбцов с расстояниями
Обрабатываю : D:\PD\Patient100\r0\mediapipe\NEUTRAL\p1_Patient100_face_landmarks_with_EAR_final.csv
Найдено 24 столбцов с расстояниями
Обрабатываю : D:\PD\Patient100\r1\mediapipe\NEUTRAL\p1_Patient100_face_landmarks_with_EAR_final.csv
Найдено 24 столбцов с расстояниями
Обрабатываю : D:\PD\Patient102\r0\mediapipe\NEUTRAL\p1_Patient102_face_landmarks_with_EAR_final.csv
Найдено 24 столбцов с расстояниями
Обрабатываю : D:\PD\Patient103\r0\mediapipe\NEUTRAL\p1_Patient103_face_landmarks_with_EAR_final.csv
Найдено 24 столбцов с расстояниями
Обрабатываю : D:\PD\Patient103\r1\mediapipe\NEUTRAL\p1_Patient103_face_landmarks_with_EAR_final.csv
Найдено 24 столбцов с расстояниями
Обрабатываю : D:\PD\Patient104\r0\mediapipe\NEUTRAL\p1_Patien

In [34]:
import os
import glob

path = 'D:\\PD\\'
folders = os.listdir(path) #список всех файлов
# folders = ['Patient92']

for e in folders:
    for r in ['r0', 'r1', 'r2', 'r3', 'r4', 'r5']:
        pattern = os.path.join(path, e, r, 'mediapipe', 'Brows', '*_statistics.csv.csv')
        for file in glob.glob(pattern):
            new_file = file.replace('.csv', '')
            
            # Проверяем, существует ли уже файл с новым именем
            if os.path.exists(new_file):
                print(f"Файл уже существует, удаляю старый: {new_file}")
                os.remove(new_file)  # Удаляем существующий файл
            
            # Переименовываем (если файл существует, он будет перезаписан)
            os.rename(file, new_file)
            print(f"Переименован: {file} -> {new_file}")

In [21]:
path = 'D:\\PD\\'
path3 = 'D:\\PD_ML_Brows2\\'
folders = os.listdir(path)
# folders = ['Patient92']

for e in folders:
    path2 = os.path.join(path, str(e))
    path4 = os.path.join(path3, str(e))
    
    # Создаем основную папку для Healthy
    if not os.path.exists(path4):
        os.makedirs(path4)
    
    f2 = os.listdir(path2)
    for r in f2:
        if r in ['r0', 'r1', 'r2', 'r3', 'r4', 'r5']:
            # Создаем подпапку для r
            output_r_dir = os.path.join(path4, str(r))
            if not os.path.exists(output_r_dir):
                os.makedirs(output_r_dir)
            print(output_r_dir)

            CSV_PATH_p3 = os.path.join(path2, r, 'mediapipe', 'Brows2', f'p3_{e}_statistics.csv')
            CSV_NORM = os.path.join(path2, r, 'mediapipe', 'NEUTRAL', f'p1_{e}_statistics_with_EAR_final.csv')
            CSV_OUT1 = os.path.join(output_r_dir, f'p3_{e}_statistics.csv')
            # else:
            #     CSV_PATH_p4 = os.path.join(path2, r, 'mediapipe', 'Smile', f'p4_m{r[1]}_{e}_statistics.csv')
            #     CSV_PATH_p5 = os.path.join(path2, r, 'mediapipe', 'Smile', f'p5_m{r[1]}_{e}_statistics.csv')
            #     CSV_NORM = os.path.join(path2, r, 'mediapipe', 'NEUTRAL', f'p1_m{r[1]}_{e}_statistics.csv')
            #     CSV_OUT1 = os.path.join(output_r_dir, f'p4_{e}_statistics.csv')
            #     CSV_OUT2 = os.path.join(output_r_dir, f'p5_{e}_statistics.csv')
            
            # Проверяем существование файлов и обрабатываем
            if os.path.exists(CSV_NORM):
                norm = pd.read_csv(CSV_NORM)
                if os.path.exists(CSV_PATH_p3):
                    df1 = pd.read_csv(CSV_PATH_p3)
                    normalize(df1, norm, CSV_OUT1)
                    print(f"Обработан: {CSV_PATH_p3} -> {CSV_OUT1}")
            else:
                print(f"[WARNING] Файл нормировки не найден: {CSV_NORM}")

D:\PD_ML_Brows2\Patient1\r0
Обработан: D:\PD\Patient1\r0\mediapipe\Brows2\p3_Patient1_statistics.csv -> D:\PD_ML_Brows2\Patient1\r0\p3_Patient1_statistics.csv
D:\PD_ML_Brows2\Patient10\r0
Обработан: D:\PD\Patient10\r0\mediapipe\Brows2\p3_Patient10_statistics.csv -> D:\PD_ML_Brows2\Patient10\r0\p3_Patient10_statistics.csv
D:\PD_ML_Brows2\Patient100\r0
Обработан: D:\PD\Patient100\r0\mediapipe\Brows2\p3_Patient100_statistics.csv -> D:\PD_ML_Brows2\Patient100\r0\p3_Patient100_statistics.csv
D:\PD_ML_Brows2\Patient100\r1
Обработан: D:\PD\Patient100\r1\mediapipe\Brows2\p3_Patient100_statistics.csv -> D:\PD_ML_Brows2\Patient100\r1\p3_Patient100_statistics.csv
D:\PD_ML_Brows2\Patient102\r0
Обработан: D:\PD\Patient102\r0\mediapipe\Brows2\p3_Patient102_statistics.csv -> D:\PD_ML_Brows2\Patient102\r0\p3_Patient102_statistics.csv
D:\PD_ML_Brows2\Patient103\r0
Обработан: D:\PD\Patient103\r0\mediapipe\Brows2\p3_Patient103_statistics.csv -> D:\PD_ML_Brows2\Patient103\r0\p3_Patient103_statistics.csv
D

In [ ]:
path = 'D:\\HEALTHY\\'
path3 = 'D:\\HEALTHY_ML_Brows2\\'
folders = os.listdir(path)
# folders = ['Patient92']

for e in folders:
    path2 = os.path.join(path, str(e))
    path4 = os.path.join(path3, str(e))
    
    # Создаем основную папку для Healthy
    if not os.path.exists(path4):
        os.makedirs(path4)
    
    f2 = os.listdir(path2)
    for r in f2:
        if r in ['r0', 'r1', 'r2', 'r3', 'r4', 'r5']:
            # Создаем подпапку для r
            output_r_dir = os.path.join(path4, str(r))
            if not os.path.exists(output_r_dir):
                os.makedirs(output_r_dir)
            print(output_r_dir)

            # Определяем суффикс для имени файла в зависимости от r
            if r == 'r0':
                suffix = ''
            else:
                # Извлекаем номер из r (например, из 'r1' получаем '1')
                attempt_num = r[1]  # берем второй символ (цифру)
                suffix = f'_m{attempt_num}'
            
            # Формируем пути к файлам
            # CSV_PATH_p4 = os.path.join(path2, r, 'mediapipe', 'Smile', f'p4{suffix}_{e}_statistics.csv')
            CSV_PATH_p3 = os.path.join(path2, r, 'mediapipe', 'Brows2', f'p3{suffix}_{e}_statistics.csv')
            CSV_NORM = os.path.join(path2, r, 'mediapipe', 'NEUTRAL', f'p1{suffix}_{e}_statistics_with_EAR_blink_interpolated.csv')
            CSV_OUT1 = os.path.join(output_r_dir, f'p3_{e}_statistics.csv')
            
            # Проверяем существование файлов и обрабатываем
            if os.path.exists(CSV_NORM):
                norm = pd.read_csv(CSV_NORM)
                if os.path.exists(CSV_PATH_p3):
                    df1 = pd.read_csv(CSV_PATH_p3)
                    normalize(df1, norm, CSV_OUT1)
                    print(f"Обработан: {CSV_PATH_p3} -> {CSV_OUT1}")
                else:
                    print(f"[WARNING] Файл p4 не найден: {CSV_PATH_p3}")

In [2]:
import os
import pandas as pd
import numpy as np
from itertools import combinations


# ============================================================
# ОСНОВНАЯ ФУНКЦИЯ
# ============================================================

def calculate_brow_raise_triangle_area_statistics(
    input_csv_path,
    output_csv_path,
    point_groups
):
    """
    Вычисляет статистики площадей треугольников
    для упражнения "поднятие бровей".

    Площадь нормируется на distance_eyes²,
    поэтому признаки инвариантны к размеру лица.
    """

    df = pd.read_csv(input_csv_path)

    print(f"\nОбрабатываю: {input_csv_path}")

    if 'distance_eyes' not in df.columns:
        raise ValueError("Нет колонки distance_eyes")

    eye_dist_sq = df['distance_eyes'].values.astype(float) ** 2
    n_frames = len(df)

    rows = []

    for group_name, indices in point_groups.items():

        triplets = list(combinations(indices, 3))

        for i, j, k in triplets:

            try:
                ax = df[f'x_{j}'].values - df[f'x_{i}'].values
                ay = df[f'y_{j}'].values - df[f'y_{i}'].values

                bx = df[f'x_{k}'].values - df[f'x_{i}'].values
                by = df[f'y_{k}'].values - df[f'y_{i}'].values

                area = (
                    0.5 *
                    np.abs(ax * by - ay * bx)
                ) / eye_dist_sq

                rows.append({
                    'group': group_name,
                    'triplet': f'area_{i}_{j}_{k}',

                    'area_min': float(np.min(area)),
                    'area_max': float(np.max(area)),
                    'area_mean': float(np.mean(area)),
                    'area_median': float(np.median(area)),
                    'area_std': float(np.std(area)),
                    'area_range': float(np.ptp(area)),

                    'num_frames': n_frames
                })

            except KeyError:
                continue

    area_df = pd.DataFrame(rows)

    os.makedirs(os.path.dirname(output_csv_path), exist_ok=True)
    area_df.to_csv(output_csv_path, index=False)

    print(f"[OK] Сохранено: {output_csv_path}")

    return area_df


# ============================================================
# ГРУППЫ ТОЧЕК ДЛЯ ПОДНЯТИЯ БРОВЕЙ
# ============================================================

BROW_RAISE_POINT_GROUPS = {

    # Упражнение 1:
    # Поднятие бровей, лобная мышца
    'brow_raise_frontalis': [
        105, 66, 107,
        336, 296, 334,
        10
    ],

    # Упражнение 2:
    # Поднятие бровей, зона депрессора бровей / нос
    'brow_raise_brow_to_nose': [
        105, 63, 66, 107,
        336, 296, 334, 293,
        4
    ],

    # Упражнение 3:
    # Верхняя и внутренняя части бровей
    'brow_raise_inner_outer_brow': [
        52, 53, 46,
        282, 283, 276,
        107, 336
    ],

    # Упражнение 4:
    # Внутренняя бровь относительно корня носа
    'brow_raise_inner_brow_nose_root': [
        107, 55,
        336, 285,
        168
    ],
}

In [3]:
area_df = calculate_brow_raise_triangle_area_statistics(

    input_csv_path=
    r'D:\PD\Patient1\r0\mediapipe\Brows2\p3_Patient1_face_landmarks.csv',

    output_csv_path=
    r'D:\PD_ML_Brows2\Patient1\r0\p3_Patient1_brow_raise_area_statistics.csv',

    point_groups=BROW_RAISE_POINT_GROUPS
)

print(area_df.head())


Обрабатываю: D:\PD\Patient1\r0\mediapipe\Brows2\p3_Patient1_face_landmarks.csv
[OK] Сохранено: D:\PD_ML_Brows2\Patient1\r0\p3_Patient1_brow_raise_area_statistics.csv
                  group          triplet  area_min  area_max  area_mean  \
0  brow_raise_frontalis  area_105_66_107  0.000003  0.002560   0.000362   
1  brow_raise_frontalis  area_105_66_336  0.000057  0.021121   0.012667   
2  brow_raise_frontalis  area_105_66_296  0.003026  0.036100   0.023354   
3  brow_raise_frontalis  area_105_66_334  0.006067  0.048909   0.032205   
4  brow_raise_frontalis   area_105_66_10  0.047247  0.073046   0.064511   

   area_median  area_std  area_range  num_frames  
0     0.000229  0.000381    0.002557         700  
1     0.013151  0.004188    0.021064         700  
2     0.024122  0.006456    0.033074         700  
3     0.033170  0.008550    0.042842         700  
4     0.065107  0.004362    0.025798         700  


In [ ]:
# ============================================================
# ОБРАБОТКА ВСЕЙ ПАПКИ PD
# ============================================================

INPUT_ROOT = r'D:\PD'
OUTPUT_ROOT = r'D:\PD_ML_Brows2'

processed = []
missing = []
errors = []

for patient_num in range(1, 134):

    patient_name = f'Patient{patient_num}'

    for r_num in range(0, 6):

        input_csv_path = os.path.join(
            INPUT_ROOT,
            patient_name,
            f'r{r_num}',
            'mediapipe',
            'Brows2',
            f'p3_{patient_name}_face_landmarks.csv'
        )

        output_csv_path = os.path.join(
            OUTPUT_ROOT,
            patient_name,
            f'r{r_num}',
            f'p3_{patient_name}_brow_raise_area_statistics.csv'
        )

        if not os.path.exists(input_csv_path):
            print(f'[WARNING] Нет файла: {input_csv_path}')
            missing.append(input_csv_path)
            continue

        try:
            area_df = calculate_brow_raise_triangle_area_statistics(
                input_csv_path=input_csv_path,
                output_csv_path=output_csv_path,
                point_groups=BROW_RAISE_POINT_GROUPS
            )

            processed.append(output_csv_path)

        except Exception as ex:
            print(f'[ERROR] {input_csv_path}')
            print(ex)
            errors.append((input_csv_path, str(ex)))


print('\n===================================')
print(f'Обработано: {len(processed)}')
print(f'Нет файлов: {len(missing)}')
print(f'Ошибок: {len(errors)}')
print('===================================')

In [ ]:
# ============================================================
# ОБРАБОТКА ВСЕЙ ПАПКИ HEALTHY
# ============================================================

INPUT_ROOT = r'D:\HEALTHY'
OUTPUT_ROOT = r'D:\HEALTHY_ML_Brows2'

processed = []
missing = []
errors = []

for patient_num in range(1, 134):

    patient_name = f'Healthy{patient_num}'

    for r_num in range(0, 6):

        # ----------------------------------------------------
        # ИМЯ ФАЙЛА
        #
        # r0:
        # p3_Healthy1_face_landmarks.csv
        #
        # r1:
        # p3_m1_Healthy1_face_landmarks.csv
        #
        # r2:
        # p3_m2_Healthy1_face_landmarks.csv
        # ...
        # ----------------------------------------------------

        if r_num == 0:

            filename = (
                f'p3_{patient_name}_face_landmarks.csv'
            )

        else:

            filename = (
                f'p3_m{r_num}_{patient_name}_face_landmarks.csv'
            )

        # ----------------------------------------------------
        # ВХОДНОЙ CSV
        # ----------------------------------------------------

        input_csv_path = os.path.join(
            INPUT_ROOT,
            patient_name,
            f'r{r_num}',
            'mediapipe',
            'Brows2',
            filename
        )

        # ----------------------------------------------------
        # ВЫХОДНОЙ CSV
        # ----------------------------------------------------

        output_csv_path = os.path.join(
            OUTPUT_ROOT,
            patient_name,
            f'r{r_num}',
            filename.replace(
                'face_landmarks',
                'brow_raise_area_statistics'
            )
        )

        # ----------------------------------------------------
        # ПРОВЕРКА НАЛИЧИЯ
        # ----------------------------------------------------

        if not os.path.exists(input_csv_path):

            print(f'[WARNING] Нет файла: {input_csv_path}')

            missing.append(input_csv_path)

            continue

        # ----------------------------------------------------
        # ОБРАБОТКА
        # ----------------------------------------------------

        try:

            area_df = calculate_brow_raise_triangle_area_statistics(
                input_csv_path=input_csv_path,
                output_csv_path=output_csv_path,
                point_groups=BROW_RAISE_POINT_GROUPS
            )

            processed.append(output_csv_path)

        except Exception as ex:

            print(f'[ERROR] {input_csv_path}')
            print(ex)

            errors.append((input_csv_path, str(ex)))


# ============================================================
# ИТОГ
# ============================================================

print('\n===================================')
print(f'Обработано: {len(processed)}')
print(f'Нет файлов: {len(missing)}')
print(f'Ошибок: {len(errors)}')
print('===================================')